# VBC polymers (all sims, ON + OFF)

## How to run polymer MD simulations

Two neighbour density modes are compared throughout. Each bead carries a neighbour cap equal to its valency.

- **ON** (`g1`, `use_global_dens=1`). The cap limits the **total** neighbour count, so a `Vk` bead saturates at about `k` neighbours summed over all types.
- **OFF** (`g0`, `use_global_dens=0`). The cap is applied **per neighbour type**, so a `Vk` bead saturates at about `k` neighbours of *each* type and its total coordination can be larger.

Plots below are generated from the trajectories `data/polymer_<config>_g{0,1}.gsd` (configs `pure_V1`..`pure_V6`, `mix_00`..`mix_19`, with `g1` = ON and `g0` = OFF). GSDs aren't part of the repository. To regenerate them, run.

```bash
python sweep.py --system polymer --mode none -g 1   # total density cap ON
python sweep.py --system polymer --mode none -g 0   # per type density cap OFF
```

`--mode none` runs the 6 pure + 20 mixed sims sequentially on one core. `--mode cpu` and `--mode gpu` submit one SLURM job per sim. The random seed is the sim's index (0..25). A single sim is e.g. `python run_polymer.py -v 1 2 3 4 5 6 --seed 6 -g 1 -o data/polymer_mix_00_g1.gsd`.

Configurations (6 pure + 20 mixed per mode).

- **`pure_Vk`**. Every bead is type `Vk`, i.e. all share valency `k`. Six runs, `V1`..`V6`.
- **`mix_NN`**. A random mixture of all six valencies, along the chain, each bead assigned a type `V1`..`V6` at random. Twenty independent realizations, `mix_00`..`mix_19`, where `NN` is the random seed, equal to the sim index.

**VBC parameters** Repulsion `E_r = 20`, attraction `E_a = -4`, repulsion cutoff `rep_rcut = 1.0`, attraction cutoff `attr_rcut = 1.5`, density window `density_lower_rcut = 1.5` to `density_upper_rcut = 2.0`, per type valency caps (each type `Vk` is capped at its own valency `k`, so for a pair `ncut_ij` and `ncut_ji` are the valencies of the two interacting types, e.g. `3` and `5` for a `V3`, `V5` pair). Beads are linked by a harmonic backbone bond - `k = 100`, `r0 = 1.0`, with bonded pairs excluded from the pair force. Integrator is NVT Langevin with `kT = 1`, `gamma = 0.1`, `dt = 0.01`. The box has `N = 500` beads at number density `0.05`, run for `50000` steps, 20 saved frames.

**Contact statistics** - contact matrices, quantiles, and the pure sim evolution use a contact radius of `r = 1.45`, with backbone bonded neighbours excluded.

For details, see `run_polymer.py` file.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import utils

NAMES = [f'pure_V{k}' for k in range(1, 7)] + [f'mix_{m:02d}' for m in range(20)]
PURE, MIX = NAMES[:6], NAMES[6:]
GVAL = {'ON': 1, 'OFF': 0}
runs = {(n, g): utils.frames(f'data/polymer_{n}_g{GVAL[g]}.gsd') for n in NAMES for g in GVAL}
Np = {n: len(np.asarray(runs[(n, 'ON')][0].particles.velocity)) for n in NAMES}

KE, P, EDRIFT = {}, {}, {}
for key in runs:
    n, g = key
    ke, p = utils.ke_and_momentum(runs[key])
    etot = ke * Np[n] + utils.potential_energy(runs[key])
    KE[key], P[key], EDRIFT[key] = ke, p, etot - etot[0]

CM_MEAN = {(n, g): utils.contact_matrix(runs[(n, g)], 'mean', r_cut=1.45) for n in MIX for g in GVAL}
CM_Q99 = {(n, g): utils.contact_matrix(runs[(n, g)], 'q99', r_cut=1.45) for n in MIX for g in GVAL}
EVOL = {(n, g): {k: [utils.contact_matrix([fr], k, r_cut=1.45)[0, 0] for fr in runs[(n, g)]] for k in ('mean', 'q90', 'q99')} for n in PURE for g in GVAL}
TYPES = {n: list(runs[(n, 'ON')][0].particles.types) for n in MIX}


def grid(items, draw, suptitle, ncols, cell=2.6):
    nrows = (len(items) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * cell, nrows * cell), layout='constrained')
    axes = np.atleast_1d(axes).ravel()
    for k, n in enumerate(items):
        draw(axes[k], n)
        axes[k].set_title(n, fontsize=9)
    for k in range(len(items), len(axes)):
        axes[k].axis('off')
    fig.suptitle(suptitle, fontsize=13)
    plt.show()


def lines(D):
    def draw(a, n):
        a.plot(D[(n, 'ON')], '-', lw=1.2, color='tab:blue')
        a.plot(D[(n, 'OFF')], '--', lw=1.2, color='tab:red')
        a.tick_params(labelsize=6)
        a.grid(alpha=0.3)
    return draw


def heatmap_of(D, g, show_sums=False):
    def draw(a, n):
        M = D[(n, g)]
        types = TYPES[n]
        nt = len(types)
        vmax = M.max() if M.max() > 0 else 1.0
        a.imshow(M, origin='lower', cmap='viridis', vmin=0, vmax=vmax)
        for i in range(nt):
            for j in range(nt):
                a.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=7,
                       color='white' if M[i, j] < 0.55 * vmax else 'black')
        a.set_xticks(range(nt))
        a.set_xticklabels(types, fontsize=7)
        a.set_xlabel('neighbour type j', fontsize=7)
        a.set_ylabel('particle type i', fontsize=7)
        a.set_yticks(range(nt))
        a.set_yticklabels(types, fontsize=7)
        if show_sums:
            cs = M.sum(axis=1)
            for i in range(nt):
                a.text(nt, i, f'{cs[i]:.2f}', ha='center', va='center', fontsize=7, fontweight='bold', color='0.1')
            a.set_xlim(-0.5, nt + 0.5)
            a.set_xticks(list(range(nt)) + [nt])
            a.set_xticklabels(types + ['Σ'], fontsize=7)
    return draw


def evolution(a, n):
    for k, c in [('mean', 'tab:blue'), ('q90', 'tab:green'), ('q99', 'tab:red')]:
        a.plot(EVOL[(n, 'ON')][k], '-', lw=1.3, color=c)
        a.plot(EVOL[(n, 'OFF')][k], '--', lw=1.3, color=c)
    a.set_xlabel('frame', fontsize=8)
    a.set_ylabel('contacts', fontsize=8)
    a.tick_params(labelsize=7)
    a.grid(alpha=0.3)
    handles = [Line2D([0], [0], color=c, label=k) for k, c in [('mean', 'tab:blue'), ('q90', 'tab:green'), ('q99', 'tab:red')]]
    handles += [Line2D([0], [0], color='k', ls='-', label='ON'), Line2D([0], [0], color='k', ls='--', label='OFF')]
    a.legend(handles=handles, fontsize=6, ncol=2)

print(len(runs), 'trajectories (ON+OFF)')

## Average KE per particle. Solid = ON, dashed = OFF

In [ ]:
grid(NAMES, lines(KE), 'avg KE per particle vs frame  (solid = ON, dashed = OFF)', 6, cell=2.4)

## Centre of mass momentum |P|. Solid = ON, dashed = OFF

In [ ]:
grid(NAMES, lines(P), 'centre-of-mass momentum |P| vs frame  (solid = ON, dashed = OFF)', 6, cell=2.4)

## Total energy drift. Solid = ON, dashed = OFF

In [ ]:
grid(NAMES, lines(EDRIFT), 'total energy drift  E(t)-E(0)  (solid = ON, dashed = OFF)', 6, cell=2.4)

## Pure sims, contact count evolution (colours = mean / 0.9 / 0.99 quantile, solid = ON, dashed = OFF)

In [ ]:
grid(PURE, evolution, 'pure sims: contact-count evolution', 3, cell=3.4)

## Mixed sims, mean contact matrix, **ON** (Σ column shows the per type total, i.e. row sums)

In [ ]:
grid(MIX, heatmap_of(CM_MEAN, 'ON', show_sums=True), 'mixed sims: mean contact matrix (ON)', 5, cell=3.4)

## Mixed sims, mean contact matrix, **OFF** (Σ column shows the per type total, i.e. row sums)

In [ ]:
grid(MIX, heatmap_of(CM_MEAN, 'OFF', show_sums=True), 'mixed sims: mean contact matrix (OFF)', 5, cell=3.4)

## Mixed sims, 0.99 quantile contact matrix, **ON**

In [ ]:
grid(MIX, heatmap_of(CM_Q99, 'ON'), 'mixed sims: 0.99-quantile contact matrix (ON)', 5, cell=3.4)

## Mixed sims, 0.99 quantile contact matrix, **OFF**

In [ ]:
grid(MIX, heatmap_of(CM_Q99, 'OFF'), 'mixed sims: 0.99-quantile contact matrix (OFF)', 5, cell=3.4)

## Final frame coordinates (xy projection, equal aspect, box scaled), aggregation

In [ ]:
def scatter(a, n, g):
    fr = runs[(n, g)][-1]
    pos = np.asarray(fr.particles.position)
    tid = np.asarray(fr.particles.typeid)
    L = float(fr.configuration.box[0])
    a.scatter(pos[:, 0], pos[:, 1], c=tid, cmap='tab10', s=5, vmin=0, vmax=5)
    a.set_xlim(-L / 2, L / 2)
    a.set_ylim(-L / 2, L / 2)
    a.set_aspect('equal')
    a.set_xticks([])
    a.set_yticks([])


for g in ['ON', 'OFF']:
    grid(NAMES, lambda a, n, g=g: scatter(a, n, g), f'final-frame coordinates, x-y projection ({g})', 6, cell=2.4)